In [1]:
# This notebook is just to make some testing datasets to apply the neural network to 

In [33]:
import xarray as xr
import numpy as np
import glob
import pandas as pd
import scipy

In [135]:
nemo_run = 'OPM026'
year = 2020
month = '09'

In [136]:
# Filepaths required to create a dataset to apply the data to...
filepath_data_ho = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/processing_ho/'
filepath_masks = filepath_data_ho + "masks_" + nemo_run + '.nc'
filepath_nico_on_nemo = filepath_data_ho + "nico_on_nemo_" + nemo_run + '.nc'
redone_slopes_fp = filepath_nn_input + nemo_run + '_' + 'redo_slopes2' + '.nc'
# Filepath for the chosen simulation year/month
filepath_nn_input = filepath_data_ho + "nn_input_"
requested_filepath = filepath_nn_input + nemo_run + '_' + str(year) + '_' + month + '.nc'
# Filepath to save the output to
filepath_AIAI_data = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Testing_data/'
save_to_fp = filepath_AIAI_data + 'nn_apply_' + nemo_run + '_' + str(year) + '_' + month + '.csv'

In [137]:
masks = xr.open_dataset(filepath_masks)
print('You loaded:')
print(filepath_masks)
closed_cavities = masks.closed_cavities.data
masks.close()
# Load the pre-interpolated grids from file
ds_load = xr.open_dataset(filepath_nico_on_nemo)
print(filepath_nico_on_nemo)
dsREDCAV_NEMO = ds_load.dsREDCAV_NEMO.data
basins_NEMO = ds_load.basins_NEMO.data
ds_load.close()
# Some ice shelves which should potentially be joined together into one bigger ice shelf
join_ice_shelves = True
if join_ice_shelves == True:
    # Dotson and Crosson?
    #basins_NEMO[basins_NEMO == 101] = 129
    # Abbot Ice Shelf
    basins_NEMO[basins_NEMO == 109] = 143
    # George VI
    basins_NEMO[basins_NEMO == 112] = 125
    # Lambert 
    basins_NEMO[basins_NEMO == 20] = 103
basin_nos_temp = np.unique(basins_NEMO)
count = np.zeros(len(basin_nos_temp))
for i in range(len(basin_nos_temp)):
    count[i] = np.sum((basins_NEMO*masks.closed_cavities_nan) == basin_nos_temp[i])
mask_keep_nos = count != 0
basin_nos = basin_nos_temp[mask_keep_nos]

You loaded:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/processing_ho/masks_OPM026.nc
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/processing_ho/nico_on_nemo_OPM026.nc


In [138]:
redone_slopes = xr.open_dataset(redone_slopes_fp)
print('You loaded:')
print(redone_slopes_fp)
data = xr.open_dataset(requested_filepath)
print(requested_filepath)
mean_T = np.ones(basins_NEMO.shape)*np.nan
mean_S = np.ones(basins_NEMO.shape)*np.nan
std_T = np.ones(basins_NEMO.shape)*np.nan
std_S = np.ones(basins_NEMO.shape)*np.nan
for j in basin_nos:
    mask_basin = basins_NEMO*masks.closed_cavities_nan == j
    mean_T[mask_basin] = np.ones(len(data.temperature_prop.data[mask_basin])) * np.nanmean(data.temperature_prop.data[mask_basin])
    mean_S[mask_basin] = np.ones(len(data.temperature_prop.data[mask_basin])) * np.nanmean(data.salinity_prop.data[mask_basin])
    std_T[mask_basin] = np.ones(len(data.temperature_prop.data[mask_basin])) * np.nanstd(data.temperature_prop.data[mask_basin])
    std_S[mask_basin] = np.ones(len(data.temperature_prop.data[mask_basin])) * np.nanstd(data.salinity_prop.data[mask_basin])
df_total = pd.DataFrame({'lat': data.lat.data[closed_cavities == 1], 
                   'lon': data.lon.data[closed_cavities == 1], 
                   'distances_GL': data.distances_GL.data[closed_cavities == 1],
                   'distances_OO': data.distances_OO.data[closed_cavities == 1],
                   'distances_OC': data.distances_OC.data[closed_cavities == 1],
                   'temperature_prop': data.temperature_prop.data[closed_cavities == 1],
                   'salinity_prop': data.salinity_prop.data[closed_cavities ==1],
                   'melt_m_ice_per_y': data.melt_ice_per_yr.data[closed_cavities == 1],
                   'corrected_isdraft': data.corrected_isdraft.data[closed_cavities ==1],
                   'bathymetry': data.bathymetry.data[closed_cavities == 1],
                   'slope_is_lon': redone_slopes.slope_is_lon.data[closed_cavities == 1],
                   'slope_is_lat': redone_slopes.slope_is_lat.data[closed_cavities == 1],
                   'slope_ba_lon': redone_slopes.slope_ba_lon.data[closed_cavities == 1],
                   'slope_ba_lat': redone_slopes.slope_ba_lat.data[closed_cavities == 1],
                   'slope_is_across_front': redone_slopes.slope_is_across_front.data[closed_cavities == 1],
                   'slope_is_towards_front': redone_slopes.slope_is_towards_front.data[closed_cavities == 1],
                   'slope_ba_across_front': redone_slopes.slope_ba_across_front.data[closed_cavities == 1],
                   'slope_ba_towards_front': redone_slopes.slope_ba_towards_front.data[closed_cavities == 1],
                   'mean_T': mean_T[closed_cavities == 1],
                   'mean_S': mean_S[closed_cavities == 1],
                   'std_T': std_T[closed_cavities == 1],
                   'std_S': std_S[closed_cavities == 1]})
df_total = df_total[~np.isnan(df_total['temperature_prop'])]

You loaded:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/processing_ho/nn_input_OPM026_redo_slopes2.nc
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/processing_ho/nn_input_OPM026_2020_09.nc


In [139]:
df_total.to_csv(save_to_fp, index = False)
print('You saved:')
print(save_to_fp)

You saved:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Testing_data/nn_apply_OPM026_2020_09.csv
